# 最終課題

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
import json
from tqdm import tqdm

# 初期設定
base_url = "https://www.musashino-u.ac.jp/"
visited = set()
sitemap = {}
queue = [base_url]

# 同一ドメインか確認
def is_same_domain(url):
    return urlparse(url).netloc.endswith("musashino-u.ac.jp")

# PDFリンクかどうか判定
def is_pdf_link(url):
    return url.lower().endswith(".pdf")

# 図書館ページかどうか判定
def is_library_page(url):
    return urlparse(url).netloc.startswith("lib.musashino-u.ac.jp")

# クロール処理（進捗バー付き）
with tqdm(total=1, desc="クロール進行状況", unit="ページ") as pbar:
    while queue:
        url = queue.pop(0)
        if url in visited:
            continue
        visited.add(url)
        time.sleep(1)

        # 図書館ページはアクセスせずに格納
        if is_library_page(url):
            sitemap[url] = "[図書館ページ - タイトル未取得]"
            pbar.total = len(visited) + len(queue)
            pbar.update(1)
            continue

        try:
            response = requests.get(url)
            soup = BeautifulSoup(response.text, "html.parser")
            title_tag = soup.find("title")
            title = title_tag.text.strip() if title_tag else ""
            sitemap[url] = title
            pbar.total = len(visited) + len(queue)
            pbar.update(1)

            # リンクを収集
            for link in soup.find_all("a", href=True):
                href = link["href"]
                full_url = urljoin(url, href)
                if is_same_domain(full_url) and not is_pdf_link(full_url):
                    if full_url not in visited and full_url not in queue:
                        queue.append(full_url)
        except Exception as e:
            print(f"取得失敗: {url} → {e}")

# 結果表示
print(f"\n✅ 取得したページ数: {len(sitemap)} 件")
print(json.dumps(sitemap, indent=2, ensure_ascii=False))